# Road Safety ML Greece — Exploratory Analysis, Benchmarking & Waze Ablation Study

**Project**: `road-safety-ml-greece` (v1.0.0)  
**Primary Data Source**: Hellenic Statistical Authority (ELSTAT) — Survey `SDT04` (*Road Traffic Accidents resulting in Death or Injury*)

---

## Scientific Integrity & Scope
1. **Track A (Official Empirical Data)**: Analyzes verified national annual totals (`2015–2024`) and regional monthly panels (`13 NUTS-2 Regions × 60 Months`, `2021–2025`).
2. **Track B (Event-Level Pipeline & Honest Waze Ablation)**: Evaluates `DummyClassifier`, `DecisionTree`, `RandomForest`, `HistGradientBoosting`, and `XGBoost` on a strictly chronological split (`2021–2023` Train, `2024` Val, `2025` Test), enforcing `validate_no_leakage(X)` and `assert_target_immutable(y_before, y_after)`.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

nat_df = pd.read_csv(PROJECT_ROOT / 'data' / 'elstat_national_annual_2015_2024.csv')
print('Official National Totals (ELSTAT 2021-2024):')
print(nat_df[nat_df['year'] >= 2021][['year', 'total_injury_accidents', 'killed_30d', 'seriously_injured', 'slightly_injured', 'ksi_per_100_accidents']].to_string(index=False))

Official National Totals (ELSTAT 2021-2024):
 year  total_injury_accidents  killed_30d  seriously_injured  slightly_injured  ksi_per_100_accidents
 2021                   10454         624                610             11746                  11.80
 2022                   10487         654                664             11961                  12.57
 2023                   10553         646                659             12156                  12.37
 2024                   11000         664                568             12803                  11.20


## 2. The Greek Road Safety Paradox: Urban Volume vs. Rural/Island Severity
In official ELSTAT statistics, **Attica** records more than half of all injury collisions in Greece (~53.5%), yet exhibits the lowest regional KSI severity rate (~6.5%) due to lower urban impact speeds and immediate tertiary trauma hospital access. Conversely, rural and island regions (**Crete, Peloponnese, South Aegean, Central Greece**) exhibit KSI rates exceeding 18%–24%.

In [2]:
reg_df = pd.read_csv(PROJECT_ROOT / 'data' / 'elstat_regional_monthly_panel_2021_2025.csv')
reg_df = reg_df[reg_df['year'].between(2021, 2025)]
summary_reg = (
    reg_df.groupby('region', as_index=False)
    .agg(total_accidents=('total_accidents', 'sum'), ksi_accidents=('ksi_accidents', 'sum'), mean_ksi_rate=('ksi_severity_rate', 'mean'))
    .sort_values('total_accidents', ascending=False)
)
summary_reg['mean_ksi_rate'] = summary_reg['mean_ksi_rate'].round(4)
print(summary_reg.head(8).to_string(index=False))

                                region  total_accidents  ksi_accidents  mean_ksi_rate
                       Attica (Αττική)            27965           1809         0.0652
Central Macedonia (Κεντρική Μακεδονία)             8625           1001         0.1166
        Western Greece (Δυτική Ελλάδα)             2732            525         0.1917
                           Crete (Κρήτη)           2512            592         0.2328
            Peloponnese (Πελοπόννησος)             2482            545         0.2191
           South Aegean (Νότιο Αιγαίο)             1931            478         0.2471
                   Thessaly (Θεσσαλία)             1803            348         0.1938
        Central Greece (Στερεά Ελλάδα)             1552            354         0.2272


## 3. Model Benchmark & Week 4 Waze Ablation Results (2025 Held-Out Test Set)
All models are trained strictly on `2021–2023`, calibrated on `2024`, and evaluated once on `2025`.

In [3]:
bench_df = pd.read_csv(PROJECT_ROOT / 'reports' / 'benchmark_comparison.csv')
print(bench_df[['track', 'model', 'roc_auc', 'roc_auc_ci_95', 'pr_auc', 'f1', 'precision', 'recall']].to_string(index=False))

                                    track                                                      model  roc_auc    roc_auc_ci_95  pr_auc     f1  precision  recall
  Track A: Official ELSTAT Regional Panel                                 DummyClassifier (Baseline)   0.5000 [0.5000, 0.5000]  0.7885 0.8817     0.7885  1.0000
  Track A: Official ELSTAT Regional Panel                                               DecisionTree   0.9068 [0.8321, 0.9684]  0.9566 0.9647     0.9318  1.0000
  Track A: Official ELSTAT Regional Panel                                               RandomForest   0.9512 [0.9036, 0.9846]  0.9855 0.9647     0.9318  1.0000
  Track A: Official ELSTAT Regional Panel                                       HistGradientBoosting   0.9377 [0.8835, 0.9756]  0.9808 0.9565     0.9308  0.9837
  Track A: Official ELSTAT Regional Panel                                                    XGBoost   0.9377 [0.8811, 0.9775]  0.9808 0.9565     0.9308  0.9837
 Track B: Event-Level Collision Be